In [1]:
# 0_FNS

#!/usr/bin/env python3

import sys
from pathlib import Path

import pandas as pd

try:
    from tqdm.auto import tqdm
except ImportError:
    # Script still runs when tqdm is not installed.
    def tqdm(iterable, **kwargs):
        return iterable


DEFAULT_FOLDER = (
    "/Users/yerik/Documents/___Divorce_2nd_Paper/"
    "__inbox/ACCOUNTS_by_account"
)

OUTPUT_NAME = "finances_master_rebuilt.xlsx"

SKIP_SHEET_KEYWORDS = (
    "summary",
    "summaries",
    "dashboard",
    "pivot",
    "totals",
)


# -----######-----###### GET INPUT FOLDER -----######-----######

def _concat_2007_folder_GET_path() -> Path:
    """
    Get the account-files folder from a command-line argument
    or from user input.
    """

    if len(sys.argv) > 1:
        folder = sys.argv[1]
    else:
        folder = input(
            f"Folder with account files [{DEFAULT_FOLDER}]: "
        ).strip()

        if not folder:
            folder = DEFAULT_FOLDER

    return Path(folder).expanduser().resolve()


# -----######-----###### FIND DATE COLUMN -----######-----######

def _concat_2007_date_GET_column(
    df: pd.DataFrame,
) -> str | None:
    """
    Find the most likely transaction date column.
    """

    date_candidates = {
        "date",
        "transaction date",
        "transaction_date",
        "posted date",
        "posted_date",
        "activity date",
        "activity_date",
    }

    for column in df.columns:
        normalized = (
            str(column)
            .strip()
            .lower()
            .replace("-", " ")
        )

        if normalized in date_candidates:
            return column

    return None


# -----######-----###### CHECK SKIPPED SHEETS -----######-----######

def _concat_2007_sheet_GET_skip(
    sheet_name: str,
) -> bool:
    """
    Return True when the worksheet looks like a summary sheet.
    """

    normalized = str(sheet_name).strip().lower()

    return any(
        keyword in normalized
        for keyword in SKIP_SHEET_KEYWORDS
    )


# -----######-----###### MAIN IMPORTABLE FUNCTION -----######-----######

def _concat_2007_i1_GET_master(
    folder_path: str | Path = DEFAULT_FOLDER,
    output_name: str = OUTPUT_NAME,
) -> pd.DataFrame:
    """
    Read all usable worksheets from every .xlsx file and create
    one master transaction workbook.

    Parameters
    ----------
    folder_path : str | Path
        Folder containing the per-account Excel files.

    output_name : str
        Name of the final merged workbook.

    Returns
    -------
    pd.DataFrame
        Combined master transaction table.
    """

    folder = Path(folder_path).expanduser().resolve()

    if not folder.is_dir():
        raise NotADirectoryError(
            f"Folder does not exist:\n{folder}"
        )

    output_path = folder / output_name

    # Find all Excel files while excluding output/temp/stash files.
    files = sorted(
        file_path
        for file_path in folder.rglob("*.xlsx")
        if file_path.resolve() != output_path.resolve()
        and not file_path.name.startswith("~$")
        and "_to_delete" not in {
            part.lower()
            for part in file_path.parts
        }
    )

    if not files:
        raise FileNotFoundError(
            f"No .xlsx files found inside:\n{folder}"
        )

    frames = []
    skipped = []
    loaded_sheets = 0

    print(f"\nScanning folder:\n{folder}\n")

    for file_path in tqdm(
        files,
        desc="Reading Excel files",
        unit="file",
    ):
        relative_file = file_path.relative_to(folder)

        try:
            excel_file = pd.ExcelFile(
                file_path,
                engine="openpyxl",
            )
        except Exception as error:
            skipped.append(
                {
                    "source_file": str(relative_file),
                    "source_sheet": None,
                    "reason": f"Workbook read error: {error}",
                }
            )
            continue

        for sheet_name in excel_file.sheet_names:

            if _concat_2007_sheet_GET_skip(sheet_name):
                skipped.append(
                    {
                        "source_file": str(relative_file),
                        "source_sheet": sheet_name,
                        "reason": "Summary sheet skipped",
                    }
                )
                continue

            try:
                df_sheet = pd.read_excel(
                    excel_file,
                    sheet_name=sheet_name,
                )
            except Exception as error:
                skipped.append(
                    {
                        "source_file": str(relative_file),
                        "source_sheet": sheet_name,
                        "reason": f"Sheet read error: {error}",
                    }
                )
                continue

            if df_sheet.empty:
                skipped.append(
                    {
                        "source_file": str(relative_file),
                        "source_sheet": sheet_name,
                        "reason": "Empty sheet",
                    }
                )
                continue

            # Clean column-name whitespace.
            df_sheet.columns = [
                str(column).strip()
                for column in df_sheet.columns
            ]

            date_column = _concat_2007_date_GET_column(
                df_sheet
            )

            if date_column is None:
                skipped.append(
                    {
                        "source_file": str(relative_file),
                        "source_sheet": sheet_name,
                        "reason": "No recognized date column",
                    }
                )
                continue

            if date_column != "date":
                df_sheet = df_sheet.rename(
                    columns={
                        date_column: "date",
                    }
                )

            df_sheet["date"] = pd.to_datetime(
                df_sheet["date"],
                errors="coerce",
            )

            # Remove fully blank rows.
            df_sheet = df_sheet.dropna(
                how="all"
            ).copy()

            # Keep provenance for every transaction.
            df_sheet["source_file"] = str(relative_file)
            df_sheet["source_sheet"] = str(sheet_name)

            frames.append(df_sheet)
            loaded_sheets += 1

            tqdm.write(
                f"+ {relative_file} | "
                f"{sheet_name} | "
                f"{len(df_sheet):,} rows"
            )

    if not frames:
        raise ValueError(
            "No usable worksheets were found. "
            "Every sheet was empty, unreadable, a summary, "
            "or missing a date column."
        )

    master = pd.concat(
        frames,
        ignore_index=True,
        sort=False,
    )

    # Sort dated records first and undated records last.
    master = (
        master
        .sort_values(
            by="date",
            ascending=True,
            na_position="last",
            kind="stable",
        )
        .reset_index(drop=True)
    )

    before_duplicates = len(master)

    # Removes only completely identical rows.
    master = (
        master
        .drop_duplicates()
        .reset_index(drop=True)
    )

    duplicates_removed = (
        before_duplicates - len(master)
    )

    # Put useful columns first when they exist.
    preferred_columns = [
        "date",
        "year",
        "month",
        "bank",
        "account",
        "account_type",
        "direction",
        "amount",
        "balance",
        "currency",
        "fees",
        "category",
        "subcategory",
        "counterparty",
        "description",
        "status",
        "cross_case",
        "txn_id",
        "source_file",
        "source_year",
        "source_sheet",
    ]

    first_columns = [
        column
        for column in preferred_columns
        if column in master.columns
    ]

    remaining_columns = [
        column
        for column in master.columns
        if column not in first_columns
    ]

    master = master[
        first_columns + remaining_columns
    ]

    with pd.ExcelWriter(
        output_path,
        engine="openpyxl",
        date_format="YYYY-MM-DD",
        datetime_format="YYYY-MM-DD",
    ) as writer:

        master.to_excel(
            writer,
            index=False,
            sheet_name="ALL_TRANSACTIONS",
        )

        worksheet = writer.sheets["ALL_TRANSACTIONS"]

        worksheet.freeze_panes = "A2"
        worksheet.auto_filter.ref = worksheet.dimensions

        # Reasonable Excel column widths.
        for column_cells in worksheet.columns:
            column_letter = column_cells[0].column_letter

            max_length = max(
                len(str(cell.value))
                if cell.value is not None
                else 0
                for cell in column_cells[:500]
            )

            worksheet.column_dimensions[
                column_letter
            ].width = min(
                max(max_length + 2, 12),
                45,
            )

        if skipped:
            df_skipped = pd.DataFrame(skipped)

            df_skipped.to_excel(
                writer,
                index=False,
                sheet_name="SKIPPED_SHEETS",
            )

            skipped_worksheet = writer.sheets[
                "SKIPPED_SHEETS"
            ]

            skipped_worksheet.freeze_panes = "A2"
            skipped_worksheet.auto_filter.ref = (
                skipped_worksheet.dimensions
            )

    print("\n" + "=" * 70)
    print(f"Files scanned:        {len(files):,}")
    print(f"Sheets loaded:        {loaded_sheets:,}")
    print(f"Master rows:          {len(master):,}")
    print(f"Duplicates removed:   {duplicates_removed:,}")
    print(f"Skipped sheets:       {len(skipped):,}")
    print(f"Saved workbook:       {output_path}")
    print("=" * 70)

    return master

In [2]:
#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!

from pathlib import Path

selected_folder = Path(
    "/Users/yerik/Documents/___Divorce_2nd_Paper/__inbox/ACCOUNTS_by_account"
).expanduser()

df_all = _concat_2007_i1_GET_master(
    folder_path=selected_folder,
    output_name=OUTPUT_NAME,
)


Scanning folder:
/Users/yerik/Documents/___Divorce_2nd_Paper/__inbox/ACCOUNTS_by_account



Reading Excel files:   0%|          | 0/19 [00:00<?, ?file/s]

+ Banco_Inbursa_Mexico_closed/transactions.xlsx | Sheet1 | 1 rows
+ Cash_App_personal/transactions.xlsx | Sheet1 | 18 rows
+ Chase_Freedom_Visa_x3262/transactions.xlsx | Sheet1 | 41 rows
+ Chase_Savings_x8635/transactions.xlsx | Sheet1 | 4 rows
+ Chase_Total_Checking_x4197/transactions.xlsx | Sheet1 | 58 rows
+ Coinbase_disclose_only/transactions.xlsx | Sheet1 | 259 rows
+ OUCU_Grizzly_Saver_03/transactions.xlsx | Sheet1 | 108 rows
+ OUCU_Platinum_Visa_x51/transactions.xlsx | Sheet1 | 476 rows
+ OUCU_Share_Secured_70/transactions.xlsx | Sheet1 | 94 rows
+ OUCU_Totally_Gold_Checking_83/transactions.xlsx | Sheet1 | 3,249 rows
+ PNC_Joint_Spend_x8775/transactions.xlsx | Sheet1 | 931 rows
+ PayPal_CAI_business/transactions.xlsx | Sheet1 | 4 rows
+ PayPal_personal/transactions.xlsx | Sheet1 | 1,256 rows
+ Robinhood_x705587533/transactions.xlsx | Sheet1 | 204 rows
+ Venmo_lizardini/transactions.xlsx | Sheet1 | 102 rows

Files scanned:        19
Sheets loaded:        15
Master rows:          

In [5]:
len(df_all)

6477

In [6]:
df_all

,date,year,month,bank,account,account_type,direction,amount,balance,currency,...,description,status,cross_case,txn_id,source_file,source_year,source_sheet,sub_account,quantity,unit_price
0,2020-12-06,2021,2021-01,OUCU,OUCU Platinum Visa,credit,out,-21.72,NaN,USD,...,24692160341100380975277 UNITEDHEALTHONE 800-65...,Posted,NaN,1206,OUCU_Platinum_Visa_x51/OUCU_Platinum_Visa_x51....,2021,Sheet1,NaN,NaN,NaN
1,2020-12-07,2021,2021-01,OUCU,OUCU Platinum Visa,credit,out,-30.49,NaN,USD,...,24427330342730240632179 MARKET SQUARE BIRMINGH...,Posted,NaN,1208,OUCU_Platinum_Visa_x51/OUCU_Platinum_Visa_x51....,2021,Sheet1,NaN,NaN,NaN
2,2020-12-08,2021,2021-01,OUCU,OUCU Platinum Visa,credit,out,-77.00,NaN,USD,...,Cash Advance ComputerLine Transfer To Share 83,Posted,NaN,1208,OUCU_Platinum_Visa_x51/OUCU_Platinum_Visa_x51....,2021,Sheet1,NaN,NaN,NaN
3,2020-12-08,2021,2021-01,OUCU,OUCU Platinum Visa,credit,out,-11.72,NaN,USD,...,Cash Advance ComputerLine Transfer To Share 83,Posted,NaN,1208,OUCU_Platinum_Visa_x51/OUCU_Platinum_Visa_x51....,2021,Sheet1,NaN,NaN,NaN
4,2020-12-26,2021,2021-01,OUCU,OUCU Platinum Visa,credit,out,-14.95,NaN,USD,...,24692160361100926597689 Audible*KO9HA2AP3 Amzn...,Posted,NaN,1226,OUCU_Platinum_Visa_x51/OUCU_Platinum_Visa_x51....,2021,Sheet1,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6472,2026-07-15,2026,7,Chase,Chase Total Checking (x4197),checking,out,-150.00,1576.76,USD,...,Zelle payment to Reynier,posted,NaN,NaN,Chase_Total_Checking_x4197/Chase_Total_Checkin...,2026,Sheet1,NaN,NaN,NaN
6473,2026-07-15,2026,7,OUCU,OUCU Totally Gold Checking (542096-83),checking,out,-32.92,708.57,USD,...,POS UBR* Uber,posted,NaN,NaN,OUCU_Totally_Gold_Checking_83/OUCU_Totally_Gol...,2026,Sheet1,NaN,NaN,NaN
6474,2026-07-15,2026,7,OUCU,OUCU Totally Gold Checking (542096-83),checking,out,-60.00,648.57,USD,...,ACH Withdrawal VENMO payment,posted,NaN,NaN,OUCU_Totally_Gold_Checking_83/OUCU_Totally_Gol...,2026,Sheet1,NaN,NaN,NaN
6475,2026-07-15,2026,7,OUCU,OUCU Totally Gold Checking (542096-83),checking,out,-291.97,356.6,USD,...,Card ••2906 online payment (biller not shown; ...,posted,NaN,NaN,OUCU_Totally_Gold_Checking_83/OUCU_Totally_Gol...,2026,Sheet1,NaN,NaN,NaN


In [3]:
# 0_FNS

from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm


# -----######-----###### PERMANENT ACCOUNT IDS -----######-----######

FOLDER_ID_MAP = {
    "Banco_Inbursa_Mexico_closed": {
        "registry_id": "A13",
        "account_uid": "ACC_INBURSA_MX_CLOSED",
    },
    "Cash_App_personal": {
        "registry_id": "A12",
        "account_uid": "ACC_CASHAPP_PERSONAL",
    },
    "Chase_Freedom_Visa_x3262": {
        "registry_id": "A07",
        "account_uid": "ACC_CHASE_FREEDOM_3262",
    },
    "Chase_Savings_x8635": {
        "registry_id": "A06",
        "account_uid": "ACC_CHASE_SAVINGS_8635",
    },
    "Chase_Total_Checking_x4197": {
        "registry_id": "A05",
        "account_uid": "ACC_CHASE_CHECKING_4197",
    },
    "Coinbase_disclose_only": {
        "registry_id": "A15",
        "account_uid": "ACC_COINBASE",
    },
    "HSBC_Mexico_closed": {
        "registry_id": "A16",
        "account_uid": "ACC_HSBC_MX_CLOSED",
    },
    "OUCU_Grizzly_Saver_03": {
        "registry_id": "A03",
        "account_uid": "ACC_OUCU_GRIZZLY_03",
    },
    "OUCU_Platinum_Visa_x51": {
        "registry_id": "A04",
        "account_uid": "ACC_OUCU_VISA_51",
    },
    "OUCU_Share_Secured_70": {
        "registry_id": "A02",
        "account_uid": "ACC_OUCU_SHARE_SECURED_70",
    },
    "OUCU_Totally_Gold_Checking_83": {
        "registry_id": "A01",
        "account_uid": "ACC_OUCU_CHECKING_83",
    },
    "PayPal_CAI_business": {
        "registry_id": "A14",
        "account_uid": "ACC_PAYPAL_CAI_BUSINESS",
    },
    "PayPal_personal": {
        "registry_id": "A09",
        "account_uid": "ACC_PAYPAL_PERSONAL",
    },
    "PNC_Joint_Spend_x8775": {
        "registry_id": "A08",
        "account_uid": "ACC_PNC_JOINT_8775",
    },
    "Robinhood_x705587533": {
        "registry_id": "A10",
        "account_uid": "ACC_ROBINHOOD",
    },
    "Venmo_lizardini": {
        "registry_id": "A11",
        "account_uid": "ACC_VENMO_LIZARDINI",
    },
}


# -----######-----###### MAIN IMPORTABLE FUNCTION -----######-----######

def _accountid_2007_i2_GET_df_ids(
    df: pd.DataFrame,
    source_file_col: str = "source_file",
    strict: bool = True,
) -> pd.DataFrame:
    """
    Add only these permanent identifiers:

        registry_id
        account_uid

    The account folder is extracted from source_file.

    Example source_file:
        Chase_Freedom_Visa_x3262/transactions.xlsx

    Parameters
    ----------
    df : pd.DataFrame
        Master transaction dataframe.

    source_file_col : str
        Column containing the relative or absolute source-file path.

    strict : bool
        True:
            Stop when an unknown folder is found.

        False:
            Leave unknown IDs blank and print the missing folders.

    Returns
    -------
    pd.DataFrame
        Original dataframe with:
            source_folder
            registry_id
            account_uid
    """

    if source_file_col not in df.columns:
        raise ValueError(
            f"Missing required column: {source_file_col}"
        )

    df_out = df.copy()

    source_folders = []

    for source_file in tqdm(
        df_out[source_file_col],
        desc="Assigning account IDs",
        unit="row",
    ):
        if pd.isna(source_file):
            source_folders.append(pd.NA)
            continue

        clean_path = str(source_file).strip().replace("\\", "/")
        source_path = Path(clean_path)

        folder_name = source_path.parent.name

        source_folders.append(
            folder_name if folder_name else pd.NA
        )

    df_out["source_folder"] = source_folders

    df_out["registry_id"] = df_out["source_folder"].map(
        lambda folder: (
            FOLDER_ID_MAP.get(folder, {}).get("registry_id")
            if pd.notna(folder)
            else pd.NA
        )
    )

    df_out["account_uid"] = df_out["source_folder"].map(
        lambda folder: (
            FOLDER_ID_MAP.get(folder, {}).get("account_uid")
            if pd.notna(folder)
            else pd.NA
        )
    )

    missing_folders = sorted(
        df_out.loc[
            df_out["source_folder"].notna()
            & df_out["registry_id"].isna(),
            "source_folder",
        ]
        .astype(str)
        .unique()
        .tolist()
    )

    if missing_folders:
        missing_message = (
            "Folders missing from FOLDER_ID_MAP:\n"
            + "\n".join(
                f"  - {folder}"
                for folder in missing_folders
            )
        )

        if strict:
            raise ValueError(missing_message)

        print(missing_message)

    # Place identifiers near the beginning.
    id_columns = [
        "registry_id",
        "account_uid",
        "source_folder",
    ]

    remaining_columns = [
        column
        for column in df_out.columns
        if column not in id_columns
    ]

    df_out = df_out[
        id_columns + remaining_columns
    ]

    return df_out

In [4]:
#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!

df_all = _accountid_2007_i2_GET_df_ids(
    df=df_all,
    source_file_col="source_file",
    strict=True,
)

df_all[
    [
        "source_folder",
        "registry_id",
        "account_uid",
    ]
].drop_duplicates().sort_values(
    "registry_id"
)

Assigning account IDs:   0%|          | 0/6741 [00:00<?, ?row/s]

,source_folder,registry_id,account_uid
6,OUCU_Totally_Gold_Checking_83,A01,ACC_OUCU_CHECKING_83
100,OUCU_Share_Secured_70,A02,ACC_OUCU_SHARE_SECURED_70
3559,OUCU_Grizzly_Saver_03,A03,ACC_OUCU_GRIZZLY_03
0,OUCU_Platinum_Visa_x51,A04,ACC_OUCU_VISA_51
6050,Chase_Total_Checking_x4197,A05,ACC_CHASE_CHECKING_4197
6079,Chase_Savings_x8635,A06,ACC_CHASE_SAVINGS_8635
6102,Chase_Freedom_Visa_x3262,A07,ACC_CHASE_FREEDOM_3262
597,PNC_Joint_Spend_x8775,A08,ACC_PNC_JOINT_8775
18,PayPal_personal,A09,ACC_PAYPAL_PERSONAL
890,Robinhood_x705587533,A10,ACC_ROBINHOOD


# tables 

In [5]:
# 0_FNS

from pathlib import Path
import re

import pandas as pd
from tqdm.auto import tqdm


# -----######-----###### CLEAN BALANCE VALUES -----######-----######

def _balance_2007_clean_GET_numeric(
    series: pd.Series,
) -> pd.Series:
    """
    Convert balances such as:

        1,465.88
        $1,465.88
        -$712.83
        (712.83)

    into numeric values.
    """

    clean = series.astype("string").str.strip()

    parentheses_negative = (
        clean.str.startswith("(", na=False)
        & clean.str.endswith(")", na=False)
    )

    clean = (
        clean
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False)
        .str.replace("(", "", regex=False)
        .str.replace(")", "", regex=False)
        .str.strip()
    )

    numeric = pd.to_numeric(
        clean,
        errors="coerce",
    )

    numeric.loc[parentheses_negative] = (
        -numeric.loc[parentheses_negative].abs()
    )

    return numeric


# -----######-----###### SAFE FILE NAME -----######-----######

def _balance_2007_name_GET_safe(
    value: str,
) -> str:
    """
    Convert an account UID into a safe filename.
    """

    safe_name = re.sub(
        pattern=r"[^A-Za-z0-9_-]+",
        repl="_",
        string=str(value).strip(),
    )

    safe_name = re.sub(
        pattern=r"_+",
        repl="_",
        string=safe_name,
    ).strip("_")

    return safe_name or "UNKNOWN_ACCOUNT"


# -----######-----###### EXCEL FORMATTING -----######-----######

def _balance_2007_excel_GET_format(
    worksheet,
) -> None:
    """
    Apply basic readable formatting to an exported worksheet.
    """

    worksheet.freeze_panes = "A2"
    worksheet.auto_filter.ref = worksheet.dimensions

    for column_cells in worksheet.columns:

        column_letter = column_cells[0].column_letter

        max_length = max(
            (
                len(str(cell.value))
                if cell.value is not None
                else 0
            )
            for cell in column_cells[:500]
        )

        worksheet.column_dimensions[
            column_letter
        ].width = min(
            max(max_length + 2, 12),
            45,
        )

    header_map = {
        cell.value: cell.column
        for cell in worksheet[1]
    }

    if "balance" in header_map:
        balance_column = header_map["balance"]

        for row_number in range(
            2,
            worksheet.max_row + 1,
        ):
            worksheet.cell(
                row=row_number,
                column=balance_column,
            ).number_format = '#,##0.00;[Red]-#,##0.00'

    date_columns = [
        "reference_date",
        "target_date",
        "balance_date_used",
        "available_from",
        "available_through",
    ]

    for date_column_name in date_columns:

        if date_column_name not in header_map:
            continue

        date_column_number = header_map[
            date_column_name
        ]

        for row_number in range(
            2,
            worksheet.max_row + 1,
        ):
            worksheet.cell(
                row=row_number,
                column=date_column_number,
            ).number_format = "yyyy-mm-dd"


# -----######-----###### MAIN IMPORTABLE FUNCTION -----######-----######

def _balance_2007_i4_GET_uid_exports(
    df: pd.DataFrame,
    uid_list: list[str] | tuple[str, ...] | str,
    reference_date: str = "2026-07-01",
    output_folder: str | Path | None = None,
) -> tuple[pd.DataFrame, dict]:
    """
    Export historical balance snapshots for each requested account UID.

    Snapshot dates:
        Reference date
        3 months before
        6 months before
        12 months before

    Every requested date is forced to the first day of the month.

    Balance selection:
        1. Use the final valid balance recorded exactly on target date.
        2. Otherwise use the final valid balance before target date.
        3. Never use a transaction after target date.
        4. If no earlier balance exists, return a blank balance with
           an explanatory status.

    Parameters
    ----------
    df : pd.DataFrame
        Master transaction dataframe.

    uid_list : list[str] | tuple[str, ...] | str
        One or more account_uid values.

    reference_date : str
        Anchor date. Default is 2026-07-01.

    output_folder : str | Path | None
        Destination folder.

        When None, creates:
            uid_balance_exports_2026-07-01

        inside the current working directory.

    Returns
    -------
    tuple[pd.DataFrame, dict]

        First value:
            Combined snapshot dataframe.

        Second value:
            Dictionary of UID to exported workbook path.
    """

    required_columns = {
        "account_uid",
        "date",
        "balance",
    }

    missing_columns = required_columns.difference(
        df.columns
    )

    if missing_columns:
        raise ValueError(
            f"Missing required columns: "
            f"{sorted(missing_columns)}"
        )

    # Accept one UID string or a list of UIDs.
    if isinstance(uid_list, str):
        uid_values = [uid_list]
    else:
        uid_values = list(uid_list)

    uid_values = [
        str(uid).strip()
        for uid in uid_values
        if pd.notna(uid)
        and str(uid).strip()
    ]

    # Remove duplicate UIDs while preserving order.
    uid_values = list(
        dict.fromkeys(uid_values)
    )

    if not uid_values:
        raise ValueError(
            "uid_list does not contain any valid account UIDs."
        )

    reference_timestamp = pd.to_datetime(
        reference_date,
        errors="coerce",
    )

    if pd.isna(reference_timestamp):
        raise ValueError(
            "reference_date must be a valid date, "
            "for example '2026-07-01'."
        )

    # Force the anchor to the first day of its month.
    reference_timestamp = (
        reference_timestamp
        .normalize()
        .replace(day=1)
    )

    reference_date_string = (
        reference_timestamp.strftime("%Y-%m-%d")
    )

    if output_folder is None:
        export_folder = Path.cwd() / (
            f"uid_balance_exports_"
            f"{reference_date_string}"
        )
    else:
        export_folder = Path(
            output_folder
        ).expanduser()

    export_folder.mkdir(
        parents=True,
        exist_ok=True,
    )

    df_work = df.copy()

    preparation_bar = tqdm(
        total=5,
        desc="Preparing master transactions",
        unit="step",
    )

    # Preserve original row order for same-day transactions.
    df_work["_original_row_order"] = range(
        len(df_work)
    )
    preparation_bar.update(1)

    df_work["account_uid"] = (
        df_work["account_uid"]
        .astype("string")
        .str.strip()
    )
    preparation_bar.update(1)

    df_work["date"] = pd.to_datetime(
        df_work["date"],
        errors="coerce",
    )

    df_work["_date_day"] = (
        df_work["date"]
        .dt.normalize()
    )
    preparation_bar.update(1)

    df_work["_balance_numeric"] = (
        _balance_2007_clean_GET_numeric(
            df_work["balance"]
        )
    )
    preparation_bar.update(1)

    df_work = df_work.sort_values(
        by=[
            "account_uid",
            "_date_day",
            "_original_row_order",
        ],
        ascending=True,
        na_position="last",
        kind="stable",
    ).reset_index(drop=True)

    preparation_bar.update(1)
    preparation_bar.close()

    target_periods = [
        {
            "period_order": 1,
            "period": "Reference",
            "months_before": 0,
        },
        {
            "period_order": 2,
            "period": "3 months before",
            "months_before": 3,
        },
        {
            "period_order": 3,
            "period": "6 months before",
            "months_before": 6,
        },
        {
            "period_order": 4,
            "period": "1 year before",
            "months_before": 12,
        },
    ]

    combined_frames = []
    exported_files = {}

    for account_uid in tqdm(
        uid_values,
        desc="Exporting account balances",
        unit="account",
    ):

        df_uid = df_work.loc[
            df_work["account_uid"].eq(
                account_uid
            )
        ].copy()

        account_exists = not df_uid.empty

        valid_date_rows = df_uid.loc[
            df_uid["_date_day"].notna()
        ].copy()

        valid_balance_rows = df_uid.loc[
            df_uid["_date_day"].notna()
            & df_uid["_balance_numeric"].notna()
        ].copy()

        if not valid_date_rows.empty:
            available_from = (
                valid_date_rows["_date_day"].min()
            )

            available_through = (
                valid_date_rows["_date_day"].max()
            )
        else:
            available_from = pd.NaT
            available_through = pd.NaT

        # General metadata fallback from the latest account row.
        if account_exists:
            metadata_row = df_uid.iloc[-1]
        else:
            metadata_row = None

        snapshot_rows = []

        for target_info in target_periods:

            target_date = (
                reference_timestamp
                - pd.DateOffset(
                    months=target_info[
                        "months_before"
                    ]
                )
            ).replace(day=1)

            selected_row = None
            selection_method = None
            status = None

            if not account_exists:
                status = "ACCOUNT_UID_NOT_FOUND"

            elif valid_date_rows.empty:
                status = "NO_VALID_TRANSACTION_DATES"

            elif valid_balance_rows.empty:
                status = "NO_VALID_BALANCE_VALUES"

            else:
                exact_rows = valid_balance_rows.loc[
                    valid_balance_rows[
                        "_date_day"
                    ].eq(target_date)
                ]

                if not exact_rows.empty:
                    # Final transaction recorded on the target date.
                    selected_row = exact_rows.iloc[-1]

                    selection_method = (
                        "EXACT_FIRST_OF_MONTH"
                    )

                    status = "BALANCE_FOUND"

                else:
                    earlier_rows = (
                        valid_balance_rows.loc[
                            valid_balance_rows[
                                "_date_day"
                            ].lt(target_date)
                        ]
                    )

                    if not earlier_rows.empty:
                        selected_row = (
                            earlier_rows.iloc[-1]
                        )

                        selection_method = (
                            "LAST_BALANCE_BEFORE_TARGET"
                        )

                        status = "BALANCE_FOUND"

                    else:
                        status = (
                            "NO_BALANCE_ON_OR_BEFORE_TARGET"
                        )

            if selected_row is not None:

                balance_date_used = selected_row[
                    "_date_day"
                ]

                balance_value = selected_row[
                    "_balance_numeric"
                ]

                gap_days = (
                    target_date - balance_date_used
                ).days

                metadata_source = selected_row

            else:
                balance_date_used = pd.NaT
                balance_value = pd.NA
                gap_days = pd.NA
                metadata_source = metadata_row

            def get_metadata(
                column_name: str,
            ):
                if (
                    metadata_source is None
                    or column_name
                    not in df_uid.columns
                ):
                    return pd.NA

                return metadata_source.get(
                    column_name,
                    pd.NA,
                )

            snapshot_rows.append(
                {
                    "registry_id": get_metadata(
                        "registry_id"
                    ),
                    "account_uid": account_uid,
                    "bank": get_metadata("bank"),
                    "account": get_metadata("account"),
                    "account_type": get_metadata(
                        "account_type"
                    ),
                    "currency": get_metadata(
                        "currency"
                    ),
                    "period_order": target_info[
                        "period_order"
                    ],
                    "period": target_info["period"],
                    "reference_date": (
                        reference_timestamp
                    ),
                    "target_date": target_date,
                    "balance_date_used": (
                        balance_date_used
                    ),
                    "balance": balance_value,
                    "days_before_target": gap_days,
                    "selection_method": (
                        selection_method
                    ),
                    "status": status,
                    "available_from": available_from,
                    "available_through": (
                        available_through
                    ),
                    "source_file_used": get_metadata(
                        "source_file"
                    ),
                    "source_sheet_used": get_metadata(
                        "source_sheet"
                    ),
                    "txn_id_used": get_metadata(
                        "txn_id"
                    ),
                    "description_used": get_metadata(
                        "description"
                    ),
                }
            )

        df_uid_snapshot = pd.DataFrame(
            snapshot_rows
        ).sort_values(
            "period_order"
        ).reset_index(drop=True)

        safe_uid = _balance_2007_name_GET_safe(
            account_uid
        )

        registry_values = (
            df_uid_snapshot["registry_id"]
            .dropna()
            .astype(str)
            .str.strip()
            .unique()
            .tolist()
        )

        registry_prefix = (
            registry_values[0]
            if registry_values
            else "NO_REGISTRY"
        )

        uid_output_path = export_folder / (
            f"{registry_prefix}_"
            f"{safe_uid}_"
            f"balances_{reference_date_string}.xlsx"
        )

        with pd.ExcelWriter(
            uid_output_path,
            engine="openpyxl",
            date_format="YYYY-MM-DD",
            datetime_format="YYYY-MM-DD",
        ) as writer:

            df_uid_snapshot.to_excel(
                writer,
                index=False,
                sheet_name="BALANCE_SNAPSHOTS",
            )

            worksheet = writer.sheets[
                "BALANCE_SNAPSHOTS"
            ]

            _balance_2007_excel_GET_format(
                worksheet
            )

        exported_files[
            account_uid
        ] = str(uid_output_path)

        combined_frames.append(
            df_uid_snapshot
        )

    df_combined = pd.concat(
        combined_frames,
        ignore_index=True,
        sort=False,
    )

    df_combined = df_combined.sort_values(
        by=[
            "registry_id",
            "account_uid",
            "period_order",
        ],
        na_position="last",
        kind="stable",
    ).reset_index(drop=True)

    combined_output_path = export_folder / (
        f"ALL_UID_BALANCE_SNAPSHOTS_"
        f"{reference_date_string}.xlsx"
    )

    with pd.ExcelWriter(
        combined_output_path,
        engine="openpyxl",
        date_format="YYYY-MM-DD",
        datetime_format="YYYY-MM-DD",
    ) as writer:

        df_combined.to_excel(
            writer,
            index=False,
            sheet_name="ALL_BALANCES",
        )

        combined_worksheet = writer.sheets[
            "ALL_BALANCES"
        ]

        _balance_2007_excel_GET_format(
            combined_worksheet
        )

    exported_files[
        "_combined"
    ] = str(combined_output_path)

    print("\n" + "=" * 80)
    print("BALANCE EXPORT COMPLETE")
    print("=" * 80)
    print(
        f"Reference date:      "
        f"{reference_date_string}"
    )
    print(
        f"Accounts requested:  "
        f"{len(uid_values):,}"
    )
    print(
        f"Snapshot rows:       "
        f"{len(df_combined):,}"
    )
    print(
        f"Export folder:       "
        f"{export_folder}"
    )
    print(
        f"Combined workbook:   "
        f"{combined_output_path}"
    )
    print("=" * 80)

    return df_combined, exported_files

In [6]:
#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!

UID_LIST = [
    "ACC_OUCU_CHECKING_83",
    "ACC_OUCU_SHARE_SECURED_70",
    "ACC_OUCU_GRIZZLY_03",
    "ACC_OUCU_VISA_51",
    "ACC_CHASE_CHECKING_4197",
    "ACC_CHASE_SAVINGS_8635",
    "ACC_CHASE_FREEDOM_3262",
    "ACC_PNC_JOINT_8775",
    "ACC_PAYPAL_PERSONAL",
    "ACC_ROBINHOOD",
    "ACC_VENMO_LIZARDINI",
]

df_balance_exports, balance_export_files = (
    _balance_2007_i4_GET_uid_exports(
        df=df_all,
        uid_list=UID_LIST,
        reference_date="2026-07-01",
        output_folder=(
            "/Users/yerik/Documents/"
            "___Divorce_2nd_Paper/"
            "__inbox/ACCOUNT_BALANCE_EXPORTS"
        ),
    )
)

df_balance_exports

Preparing master transactions:   0%|          | 0/5 [00:00<?, ?step/s]

Exporting account balances:   0%|          | 0/11 [00:00<?, ?account/s]


BALANCE EXPORT COMPLETE
Reference date:      2026-07-01
Accounts requested:  11
Snapshot rows:       44
Export folder:       /Users/yerik/Documents/___Divorce_2nd_Paper/__inbox/ACCOUNT_BALANCE_EXPORTS
Combined workbook:   /Users/yerik/Documents/___Divorce_2nd_Paper/__inbox/ACCOUNT_BALANCE_EXPORTS/ALL_UID_BALANCE_SNAPSHOTS_2026-07-01.xlsx


,registry_id,account_uid,bank,account,account_type,currency,period_order,period,reference_date,target_date,...,balance,days_before_target,selection_method,status,available_from,available_through,source_file_used,source_sheet_used,txn_id_used,description_used
0,A01,ACC_OUCU_CHECKING_83,OUCU,OUCU Totally Gold Checking (542096-83),checking,USD,1,Reference,2026-07-01,2026-07-01,...,892.8,0,EXACT_FIRST_OF_MONTH,BALANCE_FOUND,2021-01-01,2026-07-15,OUCU_Totally_Gold_Checking_83/transactions.xlsx,Sheet1,NaN,"POS AMOCO Woodward Ave, Detroit MI"
1,A01,ACC_OUCU_CHECKING_83,OUCU,OUCU Checking,checking,USD,2,3 months before,2026-07-01,2026-04-01,...,240.95,0,EXACT_FIRST_OF_MONTH,BALANCE_FOUND,2021-01-01,2026-07-15,OUCU_Totally_Gold_Checking_83/transactions.xlsx,Sheet1,NaN,Withdrawal POS #1NY3XOFNQO1J POS UBER * PENDIN...
2,A01,ACC_OUCU_CHECKING_83,OUCU,OUCU Checking,checking,USD,3,6 months before,2026-07-01,2026-01-01,...,1133.67,1,LAST_BALANCE_BEFORE_TARGET,BALANCE_FOUND,2021-01-01,2026-07-15,OUCU_Totally_Gold_Checking_83/transactions.xlsx,Sheet1,NaN,Withdrawal Debit Card 12-30 241374653643007069...
3,A01,ACC_OUCU_CHECKING_83,OUCU,OUCU Checking,checking,USD,4,1 year before,2026-07-01,2025-07-01,...,1467.89,0,EXACT_FIRST_OF_MONTH,BALANCE_FOUND,2021-01-01,2026-07-15,OUCU_Totally_Gold_Checking_83/transactions.xlsx,Sheet1,NaN,Withdrawal Debit Card 07-01 246921651821010986...
4,A02,ACC_OUCU_SHARE_SECURED_70,OUCU,OUCU Checking,savings,USD,1,Reference,2026-07-01,2026-07-01,...,500.1,1,LAST_BALANCE_BEFORE_TARGET,BALANCE_FOUND,2021-01-31,2026-06-30,OUCU_Share_Secured_70/transactions.xlsx,Sheet1,NaN,Deposit Dividend 0.050% Annual Percentage Yiel...
5,A02,ACC_OUCU_SHARE_SECURED_70,OUCU,OUCU Checking,savings,USD,2,3 months before,2026-07-01,2026-04-01,...,500.04,1,LAST_BALANCE_BEFORE_TARGET,BALANCE_FOUND,2021-01-31,2026-06-30,OUCU_Share_Secured_70/transactions.xlsx,Sheet1,NaN,Deposit Dividend 0.050% Annual Percentage Yiel...
6,A02,ACC_OUCU_SHARE_SECURED_70,OUCU,OUCU Checking,savings,USD,3,6 months before,2026-07-01,2026-01-01,...,500.03,1,LAST_BALANCE_BEFORE_TARGET,BALANCE_FOUND,2021-01-31,2026-06-30,OUCU_Share_Secured_70/transactions.xlsx,Sheet1,NaN,Deposit Dividend 0.050% Annual Percentage Yiel...
7,A02,ACC_OUCU_SHARE_SECURED_70,OUCU,OUCU Checking,savings,USD,4,1 year before,2026-07-01,2025-07-01,...,500.09,1,LAST_BALANCE_BEFORE_TARGET,BALANCE_FOUND,2021-01-31,2026-06-30,OUCU_Share_Secured_70/transactions.xlsx,Sheet1,NaN,Deposit Dividend 0.050% Annual Percentage Yiel...
8,A03,ACC_OUCU_GRIZZLY_03,OUCU,OUCU Checking,savings,USD,1,Reference,2026-07-01,2026-07-01,...,5.36,92,LAST_BALANCE_BEFORE_TARGET,BALANCE_FOUND,2024-12-28,2026-03-31,OUCU_Grizzly_Saver_03/transactions.xlsx,Sheet1,NaN,Deposit Dividend 0.050% Annual Percentage Yiel...
9,A03,ACC_OUCU_GRIZZLY_03,OUCU,OUCU Checking,savings,USD,2,3 months before,2026-07-01,2026-04-01,...,5.36,1,LAST_BALANCE_BEFORE_TARGET,BALANCE_FOUND,2024-12-28,2026-03-31,OUCU_Grizzly_Saver_03/transactions.xlsx,Sheet1,NaN,Deposit Dividend 0.050% Annual Percentage Yiel...


In [7]:
# 0_FNS

from pathlib import Path
import re
from datetime import datetime

import pandas as pd
from tqdm.auto import tqdm

from openpyxl import Workbook
from openpyxl.styles import (
    Alignment,
    Border,
    Font,
    PatternFill,
    Side,
)
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.page import PageMargins


# -----######-----###### CLEAN BALANCE VALUES -----######-----######

def _legalbal_2007_money_GET_numeric(series):
    """
    Convert money values such as:

        1,465.88
        $1,465.88
        -$712.83
        (712.83)

    into numeric values.
    """

    clean = series.astype("string").str.strip()

    parenthesis_negative = (
        clean.str.startswith("(", na=False)
        & clean.str.endswith(")", na=False)
    )

    clean = (
        clean
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False)
        .str.replace("(", "", regex=False)
        .str.replace(")", "", regex=False)
        .str.strip()
    )

    numeric = pd.to_numeric(
        clean,
        errors="coerce",
    )

    numeric.loc[parenthesis_negative] = (
        -numeric.loc[parenthesis_negative].abs()
    )

    return numeric


# -----######-----###### SAFE EXCEL VALUE -----######-----######

def _legalbal_2007_value_GET_excel(value):
    """
    Convert pandas values into values openpyxl can write safely.
    """

    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except TypeError:
        pass

    if isinstance(value, pd.Timestamp):
        return value.to_pydatetime()

    if hasattr(value, "item"):
        try:
            return value.item()
        except Exception:
            pass

    return value


# -----######-----###### GET LATEST METADATA -----######-----######

def _legalbal_2007_column_GET_latest(
    df_account,
    column_name,
):
    """
    Return the latest nonblank value from a column.
    """

    if column_name not in df_account.columns:
        return pd.NA

    values = df_account[column_name].dropna()

    if values.empty:
        return pd.NA

    values = values.loc[
        values.astype(str).str.strip().ne("")
    ]

    if values.empty:
        return pd.NA

    return values.iloc[-1]


# -----######-----###### SAFE WORKSHEET NAME -----######-----######

def _legalbal_2007_name_GET_sheet(
    value,
    existing_names,
):
    """
    Create a valid and unique Excel sheet name.
    """

    clean_name = re.sub(
        pattern=r"[\[\]\*\?/\\:]",
        repl="_",
        string=str(value),
    ).strip()

    clean_name = clean_name[:31] or "ACCOUNT"

    candidate = clean_name
    counter = 2

    while candidate in existing_names:
        suffix = f"_{counter}"
        candidate = (
            clean_name[: 31 - len(suffix)]
            + suffix
        )
        counter += 1

    return candidate


# -----######-----###### FORMAL RECORD NOTE -----######-----######

def _legalbal_2007_status_GET_note(
    status,
    method,
    gap_days,
):
    """
    Convert internal selection results into formal exhibit language.
    """

    if status == "BALANCE_FOUND":

        if method == "EXACT_FIRST_OF_MONTH":
            return (
                "Balance recorded on the requested "
                "first-of-month date."
            )

        if pd.notna(gap_days):
            day_word = (
                "day"
                if int(gap_days) == 1
                else "days"
            )

            return (
                "No balance was recorded on the exact requested "
                f"date. The most recent prior recorded balance, "
                f"{int(gap_days)} {day_word} earlier, was used."
            )

        return (
            "The most recent recorded balance on or before "
            "the requested date was used."
        )

    status_notes = {
        "ACCOUNT_UID_NOT_FOUND": (
            "No transaction records were located for this account."
        ),
        "NO_VALID_TRANSACTION_DATES": (
            "Account records were located, but no usable "
            "transaction dates were available."
        ),
        "NO_VALID_BALANCE_VALUES": (
            "Account records were located, but no usable "
            "balance values were available."
        ),
        "NO_BALANCE_ON_OR_BEFORE_TARGET": (
            "No recorded balance was available on or before "
            "the requested date."
        ),
        "INVALID_FUTURE_BALANCE_DATE": (
            "The available balance record was dated after the "
            "requested date and was therefore excluded."
        ),
        "MISSING_SNAPSHOT_ROW": (
            "No snapshot record was generated for this date."
        ),
    }

    return status_notes.get(
        status,
        "Balance information was not available.",
    )


# -----######-----###### MAIN IMPORTABLE FUNCTION -----######-----######

def _legalbal_2007_i1_GET_exhibit(
    df,
    uid_list,
    reference_date="2026-07-01",
    output_folder=None,
    output_name=None,
    exhibit_title="ACCOUNT BALANCE SNAPSHOT EXHIBIT",
    case_caption="",
    folder_id_map=None,
):
    """
    Create a legal-style account balance exhibit directly from df_all.

    For every requested account UID, the function reports balances as of:

        Reference date
        Three months before
        Six months before
        One year before

    Every date is forced to the first day of the month.

    Balance-selection rule:
        1. Use the final valid balance recorded on the exact date.
        2. Otherwise use the final valid balance before that date.
        3. Never use a balance dated after the requested date.
        4. If no usable balance exists, leave the balance blank and
           provide a formal explanatory note.

    Required columns
    ----------------
    account_uid
    date
    balance

    Useful optional columns
    -----------------------
    registry_id
    bank
    account
    account_type
    currency
    source_file
    source_sheet
    txn_id

    Returns
    -------
    df_legal_summary : pd.DataFrame
        One-row-per-account legal summary.

    df_legal_details : pd.DataFrame
        Four balance rows per requested account.

    output_path : Path
        Saved legal exhibit workbook.
    """

    required_columns = {
        "account_uid",
        "date",
        "balance",
    }

    missing_columns = required_columns.difference(
        df.columns
    )

    if missing_columns:
        raise ValueError(
            "Missing required columns: "
            f"{sorted(missing_columns)}"
        )

    # Accept either one UID or a list.
    if isinstance(uid_list, str):
        uid_values = [uid_list]
    else:
        uid_values = list(uid_list)

    uid_values = [
        str(uid).strip()
        for uid in uid_values
        if pd.notna(uid)
        and str(uid).strip()
    ]

    uid_values = list(
        dict.fromkeys(uid_values)
    )

    if not uid_values:
        raise ValueError(
            "uid_list does not contain any valid account UIDs."
        )

    reference_timestamp = pd.to_datetime(
        reference_date,
        errors="coerce",
    )

    if pd.isna(reference_timestamp):
        raise ValueError(
            "reference_date must be a valid date, "
            "for example '2026-07-01'."
        )

    # Always use the first day of the reference month.
    reference_timestamp = (
        reference_timestamp
        .normalize()
        .replace(day=1)
    )

    target_dates = [
        {
            "period_order": 1,
            "period": "Current reporting date",
            "target_date": reference_timestamp,
        },
        {
            "period_order": 2,
            "period": "Three months earlier",
            "target_date": (
                reference_timestamp
                - pd.DateOffset(months=3)
            ).replace(day=1),
        },
        {
            "period_order": 3,
            "period": "Six months earlier",
            "target_date": (
                reference_timestamp
                - pd.DateOffset(months=6)
            ).replace(day=1),
        },
        {
            "period_order": 4,
            "period": "One year earlier",
            "target_date": (
                reference_timestamp
                - pd.DateOffset(months=12)
            ).replace(day=1),
        },
    ]

    if output_folder is None:
        export_folder = Path.cwd()
    else:
        export_folder = Path(
            output_folder
        ).expanduser()

    export_folder.mkdir(
        parents=True,
        exist_ok=True,
    )

    reference_string = (
        reference_timestamp.strftime("%Y-%m-%d")
    )

    if output_name is None:
        output_name = (
            "LEGAL_ACCOUNT_BALANCE_EXHIBIT_"
            f"{reference_string}.xlsx"
        )

    output_path = export_folder / output_name

    # Build UID -> registry ID lookup from FOLDER_ID_MAP.
    if folder_id_map is None:
        folder_id_map = globals().get(
            "FOLDER_ID_MAP",
            {},
        )

    registry_by_uid = {}

    for registry_record in folder_id_map.values():

        if not isinstance(
            registry_record,
            dict,
        ):
            continue

        mapped_uid = registry_record.get(
            "account_uid"
        )

        mapped_registry = registry_record.get(
            "registry_id"
        )

        if mapped_uid:
            registry_by_uid[
                str(mapped_uid).strip()
            ] = mapped_registry

    # Clean master dataframe.
    df_work = df.copy()

    preparation_bar = tqdm(
        total=5,
        desc="Preparing legal balance exhibit",
        unit="step",
    )

    df_work["_original_row_order"] = range(
        len(df_work)
    )
    preparation_bar.update(1)

    df_work["account_uid"] = (
        df_work["account_uid"]
        .astype("string")
        .str.strip()
    )
    preparation_bar.update(1)

    df_work["date"] = pd.to_datetime(
        df_work["date"],
        errors="coerce",
    )

    df_work["_date_day"] = (
        df_work["date"]
        .dt.normalize()
    )
    preparation_bar.update(1)

    df_work["_balance_numeric"] = (
        _legalbal_2007_money_GET_numeric(
            df_work["balance"]
        )
    )
    preparation_bar.update(1)

    df_work = (
        df_work
        .sort_values(
            by=[
                "account_uid",
                "_date_day",
                "_original_row_order",
            ],
            ascending=True,
            na_position="last",
            kind="stable",
        )
        .reset_index(drop=True)
    )
    preparation_bar.update(1)
    preparation_bar.close()

    detail_rows = []
    summary_rows = []

    # Process each requested account.
    for account_uid in tqdm(
        uid_values,
        desc="Building account exhibits",
        unit="account",
    ):

        df_account = df_work.loc[
            df_work["account_uid"].eq(
                account_uid
            )
        ].copy()

        account_exists = not df_account.empty

        valid_date_rows = df_account.loc[
            df_account["_date_day"].notna()
        ].copy()

        valid_balance_rows = df_account.loc[
            df_account["_date_day"].notna()
            & df_account["_balance_numeric"].notna()
        ].copy()

        registry_id = (
            _legalbal_2007_column_GET_latest(
                df_account,
                "registry_id",
            )
        )

        if pd.isna(registry_id):
            registry_id = registry_by_uid.get(
                account_uid,
                pd.NA,
            )

        bank = _legalbal_2007_column_GET_latest(
            df_account,
            "bank",
        )

        account_name = (
            _legalbal_2007_column_GET_latest(
                df_account,
                "account",
            )
        )

        account_type = (
            _legalbal_2007_column_GET_latest(
                df_account,
                "account_type",
            )
        )

        currency = (
            _legalbal_2007_column_GET_latest(
                df_account,
                "currency",
            )
        )

        if not valid_date_rows.empty:
            available_from = (
                valid_date_rows["_date_day"].min()
            )

            available_through = (
                valid_date_rows["_date_day"].max()
            )

            coverage_text = (
                f"{available_from:%m/%d/%Y} through "
                f"{available_through:%m/%d/%Y}"
            )
        else:
            available_from = pd.NaT
            available_through = pd.NaT
            coverage_text = "No usable dated records located"

        account_detail_rows = []

        for target_info in target_dates:

            target_date = target_info[
                "target_date"
            ]

            selected_row = None
            method = "UNAVAILABLE"
            status = None

            if not account_exists:
                status = "ACCOUNT_UID_NOT_FOUND"

            elif valid_date_rows.empty:
                status = "NO_VALID_TRANSACTION_DATES"

            elif valid_balance_rows.empty:
                status = "NO_VALID_BALANCE_VALUES"

            else:
                exact_rows = valid_balance_rows.loc[
                    valid_balance_rows[
                        "_date_day"
                    ].eq(target_date)
                ]

                if not exact_rows.empty:
                    selected_row = exact_rows.iloc[-1]
                    method = "EXACT_FIRST_OF_MONTH"
                    status = "BALANCE_FOUND"

                else:
                    earlier_rows = valid_balance_rows.loc[
                        valid_balance_rows[
                            "_date_day"
                        ].lt(target_date)
                    ]

                    if not earlier_rows.empty:
                        selected_row = (
                            earlier_rows.iloc[-1]
                        )
                        method = (
                            "LAST_BALANCE_BEFORE_TARGET"
                        )
                        status = "BALANCE_FOUND"

                    else:
                        status = (
                            "NO_BALANCE_ON_OR_BEFORE_TARGET"
                        )

            if selected_row is not None:

                balance_date_used = selected_row[
                    "_date_day"
                ]

                # Final protection against a future record.
                if balance_date_used > target_date:
                    balance_date_used = pd.NaT
                    balance_value = pd.NA
                    selected_row = None
                    method = "UNAVAILABLE"
                    status = (
                        "INVALID_FUTURE_BALANCE_DATE"
                    )
                    gap_days = pd.NA

                else:
                    balance_value = selected_row[
                        "_balance_numeric"
                    ]

                    gap_days = (
                        target_date
                        - balance_date_used
                    ).days

            else:
                balance_date_used = pd.NaT
                balance_value = pd.NA
                gap_days = pd.NA

            source_file = pd.NA
            source_sheet = pd.NA
            txn_id = pd.NA

            if selected_row is not None:

                if "source_file" in selected_row.index:
                    source_file = selected_row.get(
                        "source_file",
                        pd.NA,
                    )

                if "source_sheet" in selected_row.index:
                    source_sheet = selected_row.get(
                        "source_sheet",
                        pd.NA,
                    )

                if "txn_id" in selected_row.index:
                    txn_id = selected_row.get(
                        "txn_id",
                        pd.NA,
                    )

            source_parts = []

            if pd.notna(source_file):
                source_parts.append(
                    str(source_file)
                )

            if pd.notna(source_sheet):
                source_parts.append(
                    f"Sheet: {source_sheet}"
                )

            if pd.notna(txn_id):
                source_parts.append(
                    f"Transaction ID: {txn_id}"
                )

            source_record = (
                " | ".join(source_parts)
                if source_parts
                else "No specific source record available"
            )

            formal_note = (
                _legalbal_2007_status_GET_note(
                    status=status,
                    method=method,
                    gap_days=gap_days,
                )
            )

            detail_record = {
                "Registry ID": registry_id,
                "Account UID": account_uid,
                "Financial Institution": bank,
                "Account": account_name,
                "Account Type": account_type,
                "Currency": currency,
                "Period Order": target_info[
                    "period_order"
                ],
                "Period": target_info["period"],
                "Requested As-of Date": target_date,
                "Balance Date Used": (
                    balance_date_used
                ),
                "Balance": balance_value,
                "Record Method": (
                    "Exact first-of-month record"
                    if method
                    == "EXACT_FIRST_OF_MONTH"
                    else (
                        "Most recent prior record"
                        if method
                        == "LAST_BALANCE_BEFORE_TARGET"
                        else "Unavailable"
                    )
                ),
                "Record Note": formal_note,
                "Source Record": source_record,
                "Internal Status": status,
                "Available From": available_from,
                "Available Through": available_through,
            }

            detail_rows.append(
                detail_record
            )

            account_detail_rows.append(
                detail_record
            )

        found_count = sum(
            pd.notna(row["Balance"])
            for row in account_detail_rows
        )

        if found_count == 4:
            completeness = (
                "Complete — 4 of 4 balances located"
            )
        elif found_count == 0:
            completeness = (
                "No reportable balances located"
            )
        else:
            completeness = (
                f"Partial — {found_count} of 4 "
                "balances located"
            )

        summary_record = {
            "Registry ID": registry_id,
            "Financial Institution": bank,
            "Account": account_name,
            "Account UID": account_uid,
            "Account Type": account_type,
            "Currency": currency,
            "Record Coverage": coverage_text,
            "Completeness": completeness,
        }

        for account_detail in account_detail_rows:
            balance_column = (
                "Balance as of "
                + account_detail[
                    "Requested As-of Date"
                ].strftime("%m/%d/%Y")
            )

            summary_record[balance_column] = (
                account_detail["Balance"]
            )

        summary_rows.append(
            summary_record
        )

    df_legal_details = pd.DataFrame(
        detail_rows
    )

    balance_summary_columns = [
        "Balance as of "
        + target["target_date"].strftime(
            "%m/%d/%Y"
        )
        for target in target_dates
    ]

    summary_columns = [
        "Registry ID",
        "Financial Institution",
        "Account",
        "Account UID",
        "Account Type",
        "Currency",
        *balance_summary_columns,
        "Record Coverage",
        "Completeness",
    ]

    df_legal_summary = (
        pd.DataFrame(summary_rows)
        .reindex(columns=summary_columns)
        .sort_values(
            by=[
                "Registry ID",
                "Account UID",
            ],
            na_position="last",
            kind="stable",
        )
        .reset_index(drop=True)
    )

    # ---------------------------------------------------------
    # CREATE LEGAL-STYLE WORKBOOK
    # ---------------------------------------------------------

    workbook = Workbook()

    summary_sheet = workbook.active
    summary_sheet.title = "SUMMARY"
    summary_sheet.sheet_view.showGridLines = False

    dark_fill = PatternFill(
        fill_type="solid",
        fgColor="1F2937",
    )

    medium_fill = PatternFill(
        fill_type="solid",
        fgColor="D1D5DB",
    )

    light_fill = PatternFill(
        fill_type="solid",
        fgColor="F3F4F6",
    )

    white_font = Font(
        name="Arial",
        size=10,
        bold=True,
        color="FFFFFF",
    )

    title_font = Font(
        name="Arial",
        size=16,
        bold=True,
        color="111827",
    )

    subtitle_font = Font(
        name="Arial",
        size=10,
        italic=True,
        color="374151",
    )

    body_font = Font(
        name="Arial",
        size=10,
        color="111827",
    )

    label_font = Font(
        name="Arial",
        size=10,
        bold=True,
        color="111827",
    )

    thin_gray = Side(
        style="thin",
        color="9CA3AF",
    )

    table_border = Border(
        left=thin_gray,
        right=thin_gray,
        top=thin_gray,
        bottom=thin_gray,
    )

    currency_format = (
        '$#,##0.00;($#,##0.00);-'
    )

    date_format = "mm/dd/yyyy"

    # ---------------------------------------------------------
    # SUMMARY SHEET
    # ---------------------------------------------------------

    total_summary_columns = len(
        summary_columns
    )

    summary_sheet.merge_cells(
        start_row=1,
        start_column=1,
        end_row=1,
        end_column=total_summary_columns,
    )

    title_cell = summary_sheet.cell(
        row=1,
        column=1,
        value=exhibit_title,
    )

    title_cell.font = title_font
    title_cell.alignment = Alignment(
        horizontal="center",
        vertical="center",
    )

    summary_sheet.row_dimensions[1].height = 27

    summary_sheet.merge_cells(
        start_row=2,
        start_column=1,
        end_row=2,
        end_column=total_summary_columns,
    )

    subtitle_text = (
        case_caption.strip()
        if case_caption
        else (
            "Comparative account balances using the "
            "last available recorded balance on or before "
            "each requested date."
        )
    )

    subtitle_cell = summary_sheet.cell(
        row=2,
        column=1,
        value=subtitle_text,
    )

    subtitle_cell.font = subtitle_font
    subtitle_cell.alignment = Alignment(
        horizontal="center",
        vertical="center",
        wrap_text=True,
    )

    summary_sheet.merge_cells(
        start_row=3,
        start_column=1,
        end_row=3,
        end_column=total_summary_columns,
    )

    summary_sheet.cell(
        row=3,
        column=1,
        value=(
            "Reporting date: "
            f"{reference_timestamp:%B %d, %Y}"
            "  |  Prepared: "
            f"{datetime.now():%B %d, %Y}"
        ),
    ).alignment = Alignment(
        horizontal="center"
    )

    summary_sheet.cell(
        row=3,
        column=1,
    ).font = body_font

    header_row = 5

    for column_number, header in enumerate(
        summary_columns,
        start=1,
    ):
        cell = summary_sheet.cell(
            row=header_row,
            column=column_number,
            value=header,
        )

        cell.fill = dark_fill
        cell.font = white_font
        cell.border = table_border
        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True,
        )

    summary_sheet.row_dimensions[
        header_row
    ].height = 38

    for row_offset, record in enumerate(
        df_legal_summary.to_dict("records"),
        start=header_row + 1,
    ):

        for column_number, column_name in enumerate(
            summary_columns,
            start=1,
        ):
            value = _legalbal_2007_value_GET_excel(
                record.get(column_name)
            )

            cell = summary_sheet.cell(
                row=row_offset,
                column=column_number,
                value=value,
            )

            cell.font = body_font
            cell.border = table_border
            cell.alignment = Alignment(
                horizontal=(
                    "right"
                    if column_name
                    in balance_summary_columns
                    else "left"
                ),
                vertical="top",
                wrap_text=True,
            )

            if column_name in balance_summary_columns:
                cell.number_format = currency_format

        if row_offset % 2 == 0:
            for column_number in range(
                1,
                total_summary_columns + 1,
            ):
                summary_sheet.cell(
                    row=row_offset,
                    column=column_number,
                ).fill = light_fill

    summary_last_row = (
        header_row + len(df_legal_summary)
    )

    note_row = summary_last_row + 2

    summary_sheet.merge_cells(
        start_row=note_row,
        start_column=1,
        end_row=note_row + 2,
        end_column=total_summary_columns,
    )

    note_cell = summary_sheet.cell(
        row=note_row,
        column=1,
        value=(
            "Methodology: For each requested first-of-month "
            "date, the exhibit uses the final recorded balance "
            "on that exact date. When no exact-date balance is "
            "available, the final recorded balance before that "
            "date is used. A later balance is never carried "
            "backward. Blank balance cells indicate that no "
            "usable balance existed on or before the requested date."
        ),
    )

    note_cell.font = subtitle_font
    note_cell.alignment = Alignment(
        horizontal="left",
        vertical="top",
        wrap_text=True,
    )

    note_cell.fill = light_fill
    note_cell.border = table_border

    summary_widths = {
        "A": 12,
        "B": 20,
        "C": 30,
        "D": 31,
        "E": 18,
        "F": 10,
        "G": 17,
        "H": 17,
        "I": 17,
        "J": 17,
        "K": 24,
        "L": 27,
    }

    for column_letter, width in (
        summary_widths.items()
    ):
        summary_sheet.column_dimensions[
            column_letter
        ].width = width

    summary_sheet.freeze_panes = (
        f"A{header_row + 1}"
    )

    summary_sheet.auto_filter.ref = (
        f"A{header_row}:"
        f"{get_column_letter(total_summary_columns)}"
        f"{summary_last_row}"
    )

    summary_sheet.page_setup.orientation = (
        "landscape"
    )

    summary_sheet.page_setup.fitToWidth = 1
    summary_sheet.page_setup.fitToHeight = 0
    summary_sheet.sheet_properties.pageSetUpPr.fitToPage = True

    summary_sheet.page_margins = PageMargins(
        left=0.25,
        right=0.25,
        top=0.5,
        bottom=0.5,
        header=0.2,
        footer=0.2,
    )

    summary_sheet.print_title_rows = (
        f"1:{header_row}"
    )

    summary_sheet.print_area = (
        f"A1:"
        f"{get_column_letter(total_summary_columns)}"
        f"{note_row + 2}"
    )

    summary_sheet.oddFooter.left.text = (
        exhibit_title
    )

    summary_sheet.oddFooter.right.text = (
        "Page &P of &N"
    )

    # ---------------------------------------------------------
    # ACCOUNT-SPECIFIC EXHIBIT SHEETS
    # ---------------------------------------------------------

    existing_sheet_names = {
        "SUMMARY"
    }

    for account_uid in tqdm(
        uid_values,
        desc="Formatting exhibit sheets",
        unit="sheet",
    ):

        df_account_detail = (
            df_legal_details.loc[
                df_legal_details[
                    "Account UID"
                ].eq(account_uid)
            ]
            .sort_values("Period Order")
            .reset_index(drop=True)
        )

        first_record = (
            df_account_detail.iloc[0]
            if not df_account_detail.empty
            else pd.Series(dtype="object")
        )

        registry_id = first_record.get(
            "Registry ID",
            pd.NA,
        )

        sheet_base = (
            str(registry_id)
            if pd.notna(registry_id)
            else account_uid
        )

        sheet_name = (
            _legalbal_2007_name_GET_sheet(
                sheet_base,
                existing_sheet_names,
            )
        )

        existing_sheet_names.add(
            sheet_name
        )

        worksheet = workbook.create_sheet(
            title=sheet_name
        )

        worksheet.sheet_view.showGridLines = False

        worksheet.merge_cells("A1:G1")

        worksheet["A1"] = (
            f"{exhibit_title} — "
            f"{sheet_base}"
        )

        worksheet["A1"].font = title_font
        worksheet["A1"].alignment = Alignment(
            horizontal="center",
            vertical="center",
        )

        worksheet.row_dimensions[1].height = 27

        worksheet.merge_cells("A2:G2")

        worksheet["A2"] = (
            "Balance history as of "
            f"{reference_timestamp:%B %d, %Y}"
        )

        worksheet["A2"].font = subtitle_font
        worksheet["A2"].alignment = Alignment(
            horizontal="center",
            vertical="center",
        )

        metadata_pairs = [
            (
                "Registry ID",
                first_record.get(
                    "Registry ID",
                    pd.NA,
                ),
                "Account UID",
                account_uid,
            ),
            (
                "Financial Institution",
                first_record.get(
                    "Financial Institution",
                    pd.NA,
                ),
                "Account",
                first_record.get(
                    "Account",
                    pd.NA,
                ),
            ),
            (
                "Account Type",
                first_record.get(
                    "Account Type",
                    pd.NA,
                ),
                "Currency",
                first_record.get(
                    "Currency",
                    pd.NA,
                ),
            ),
        ]

        for row_number, pair in enumerate(
            metadata_pairs,
            start=4,
        ):
            left_label, left_value, right_label, right_value = pair

            worksheet.cell(
                row=row_number,
                column=1,
                value=left_label,
            )

            worksheet.merge_cells(
                start_row=row_number,
                start_column=2,
                end_row=row_number,
                end_column=3,
            )

            worksheet.cell(
                row=row_number,
                column=2,
                value=_legalbal_2007_value_GET_excel(
                    left_value
                ),
            )

            worksheet.cell(
                row=row_number,
                column=4,
                value=right_label,
            )

            worksheet.merge_cells(
                start_row=row_number,
                start_column=5,
                end_row=row_number,
                end_column=7,
            )

            worksheet.cell(
                row=row_number,
                column=5,
                value=_legalbal_2007_value_GET_excel(
                    right_value
                ),
            )

            for label_column in [1, 4]:
                label_cell = worksheet.cell(
                    row=row_number,
                    column=label_column,
                )

                label_cell.font = label_font
                label_cell.fill = medium_fill
                label_cell.border = table_border
                label_cell.alignment = Alignment(
                    vertical="center",
                )

            for value_column in [2, 5]:
                value_cell = worksheet.cell(
                    row=row_number,
                    column=value_column,
                )

                value_cell.font = body_font
                value_cell.border = table_border
                value_cell.alignment = Alignment(
                    vertical="center",
                    wrap_text=True,
                )

        worksheet["A7"] = "Record Coverage"
        worksheet["A7"].font = label_font
        worksheet["A7"].fill = medium_fill
        worksheet["A7"].border = table_border

        worksheet.merge_cells("B7:G7")

        worksheet["B7"] = (
            first_record.get(
                "Record Coverage",
                None,
            )
            if "Record Coverage"
            in first_record.index
            else (
                (
                    f"{first_record.get('Available From'):%m/%d/%Y}"
                    " through "
                    f"{first_record.get('Available Through'):%m/%d/%Y}"
                )
                if pd.notna(
                    first_record.get(
                        "Available From",
                        pd.NaT,
                    )
                )
                and pd.notna(
                    first_record.get(
                        "Available Through",
                        pd.NaT,
                    )
                )
                else "No usable dated records located"
            )
        )

        worksheet["B7"].font = body_font
        worksheet["B7"].border = table_border
        worksheet["B7"].alignment = Alignment(
            wrap_text=True,
        )

        detail_headers = [
            "Period",
            "Requested As-of Date",
            "Balance Date Used",
            "Balance",
            "Record Method",
            "Record Note",
            "Source Record",
        ]

        detail_header_row = 9

        for column_number, header in enumerate(
            detail_headers,
            start=1,
        ):
            cell = worksheet.cell(
                row=detail_header_row,
                column=column_number,
                value=header,
            )

            cell.fill = dark_fill
            cell.font = white_font
            cell.border = table_border
            cell.alignment = Alignment(
                horizontal="center",
                vertical="center",
                wrap_text=True,
            )

        worksheet.row_dimensions[
            detail_header_row
        ].height = 38

        for row_offset, detail_record in enumerate(
            df_account_detail.to_dict(
                "records"
            ),
            start=detail_header_row + 1,
        ):
            values = [
                detail_record.get("Period"),
                detail_record.get(
                    "Requested As-of Date"
                ),
                detail_record.get(
                    "Balance Date Used"
                ),
                detail_record.get("Balance"),
                detail_record.get(
                    "Record Method"
                ),
                detail_record.get(
                    "Record Note"
                ),
                detail_record.get(
                    "Source Record"
                ),
            ]

            for column_number, value in enumerate(
                values,
                start=1,
            ):
                cell = worksheet.cell(
                    row=row_offset,
                    column=column_number,
                    value=_legalbal_2007_value_GET_excel(
                        value
                    ),
                )

                cell.font = body_font
                cell.border = table_border
                cell.alignment = Alignment(
                    horizontal=(
                        "right"
                        if column_number == 4
                        else "left"
                    ),
                    vertical="top",
                    wrap_text=True,
                )

                if column_number in [2, 3]:
                    cell.number_format = date_format

                if column_number == 4:
                    cell.number_format = (
                        currency_format
                    )

            if row_offset % 2 == 0:
                for column_number in range(
                    1,
                    8,
                ):
                    worksheet.cell(
                        row=row_offset,
                        column=column_number,
                    ).fill = light_fill

            worksheet.row_dimensions[
                row_offset
            ].height = 52

        detail_last_row = (
            detail_header_row
            + len(df_account_detail)
        )

        exhibit_note_row = detail_last_row + 2

        worksheet.merge_cells(
            start_row=exhibit_note_row,
            start_column=1,
            end_row=exhibit_note_row + 2,
            end_column=7,
        )

        exhibit_note = worksheet.cell(
            row=exhibit_note_row,
            column=1,
            value=(
                "Exhibit methodology: An exact balance recorded "
                "on the requested first-of-month date is used when "
                "available. Otherwise, the most recent balance "
                "recorded before that date is shown. No later-dated "
                "balance is used to estimate an earlier date. "
                "Unavailable balances are left blank and explained "
                "in the Record Note column."
            ),
        )

        exhibit_note.font = subtitle_font
        exhibit_note.fill = light_fill
        exhibit_note.border = table_border
        exhibit_note.alignment = Alignment(
            horizontal="left",
            vertical="top",
            wrap_text=True,
        )

        account_widths = {
            "A": 24,
            "B": 19,
            "C": 19,
            "D": 16,
            "E": 24,
            "F": 51,
            "G": 48,
        }

        for column_letter, width in (
            account_widths.items()
        ):
            worksheet.column_dimensions[
                column_letter
            ].width = width

        worksheet.freeze_panes = (
            f"A{detail_header_row + 1}"
        )

        worksheet.auto_filter.ref = (
            f"A{detail_header_row}:"
            f"G{detail_last_row}"
        )

        worksheet.page_setup.orientation = (
            "landscape"
        )

        worksheet.page_setup.fitToWidth = 1
        worksheet.page_setup.fitToHeight = 1
        worksheet.sheet_properties.pageSetUpPr.fitToPage = True

        worksheet.page_margins = PageMargins(
            left=0.25,
            right=0.25,
            top=0.5,
            bottom=0.5,
            header=0.2,
            footer=0.2,
        )

        worksheet.print_title_rows = (
            f"1:{detail_header_row}"
        )

        worksheet.print_area = (
            f"A1:G{exhibit_note_row + 2}"
        )

        worksheet.oddHeader.left.text = (
            str(sheet_base)
        )

        worksheet.oddFooter.left.text = (
            exhibit_title
        )

        worksheet.oddFooter.right.text = (
            "Page &P of &N"
        )

    # ---------------------------------------------------------
    # RECORD EXCEPTIONS SHEET
    # ---------------------------------------------------------

    df_exceptions = df_legal_details.loc[
        df_legal_details["Balance"].isna()
    ].copy()

    if not df_exceptions.empty:

        exception_sheet = workbook.create_sheet(
            title="RECORD_NOTES"
        )

        exception_sheet.sheet_view.showGridLines = False

        exception_columns = [
            "Registry ID",
            "Account UID",
            "Financial Institution",
            "Account",
            "Period",
            "Requested As-of Date",
            "Record Note",
            "Internal Status",
        ]

        exception_sheet.merge_cells(
            start_row=1,
            start_column=1,
            end_row=1,
            end_column=len(exception_columns),
        )

        exception_sheet["A1"] = (
            "RECORD AVAILABILITY NOTES"
        )

        exception_sheet["A1"].font = title_font
        exception_sheet["A1"].alignment = Alignment(
            horizontal="center"
        )

        for column_number, header in enumerate(
            exception_columns,
            start=1,
        ):
            cell = exception_sheet.cell(
                row=3,
                column=column_number,
                value=header,
            )

            cell.fill = dark_fill
            cell.font = white_font
            cell.border = table_border
            cell.alignment = Alignment(
                horizontal="center",
                wrap_text=True,
            )

        for row_number, record in enumerate(
            df_exceptions[
                exception_columns
            ].to_dict("records"),
            start=4,
        ):
            for column_number, column_name in enumerate(
                exception_columns,
                start=1,
            ):
                cell = exception_sheet.cell(
                    row=row_number,
                    column=column_number,
                    value=_legalbal_2007_value_GET_excel(
                        record.get(column_name)
                    ),
                )

                cell.font = body_font
                cell.border = table_border
                cell.alignment = Alignment(
                    vertical="top",
                    wrap_text=True,
                )

                if column_name == (
                    "Requested As-of Date"
                ):
                    cell.number_format = date_format

        exception_widths = [
            12,
            31,
            20,
            30,
            23,
            19,
            55,
            30,
        ]

        for column_number, width in enumerate(
            exception_widths,
            start=1,
        ):
            exception_sheet.column_dimensions[
                get_column_letter(
                    column_number
                )
            ].width = width

        exception_sheet.freeze_panes = "A4"
        exception_sheet.page_setup.orientation = (
            "landscape"
        )

        exception_sheet.page_setup.fitToWidth = 1
        exception_sheet.sheet_properties.pageSetUpPr.fitToPage = True

    workbook.save(
        output_path
    )

    print("\n" + "=" * 80)
    print("LEGAL ACCOUNT BALANCE EXHIBIT CREATED")
    print("=" * 80)
    print(
        f"Accounts included:    "
        f"{len(uid_values):,}"
    )
    print(
        f"Summary rows:         "
        f"{len(df_legal_summary):,}"
    )
    print(
        f"Balance detail rows:  "
        f"{len(df_legal_details):,}"
    )
    print(
        f"Missing balances:     "
        f"{df_legal_details['Balance'].isna().sum():,}"
    )
    print(
        f"Saved workbook:       "
        f"{output_path}"
    )
    print("=" * 80)

    return (
        df_legal_summary,
        df_legal_details,
        output_path,
    )

In [8]:
#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!

from IPython.display import display


UID_LIST = [
    "ACC_OUCU_CHECKING_83",
    "ACC_OUCU_SHARE_SECURED_70",
    "ACC_OUCU_GRIZZLY_03",
    "ACC_OUCU_VISA_51",
    "ACC_CHASE_CHECKING_4197",
    "ACC_CHASE_SAVINGS_8635",
    "ACC_CHASE_FREEDOM_3262",
    "ACC_PNC_JOINT_8775",
    "ACC_PAYPAL_PERSONAL",
    "ACC_ROBINHOOD",
    "ACC_VENMO_LIZARDINI",
]


df_legal_summary, df_legal_details, legal_exhibit_path = (
    _legalbal_2007_i1_GET_exhibit(
        df=df_all,
        uid_list=UID_LIST,
        reference_date="2026-07-01",
        output_folder=(
            "/Users/yerik/Documents/"
            "___Divorce_2nd_Paper/"
            "__inbox/ACCOUNT_BALANCE_EXPORTS"
        ),
        exhibit_title=(
            "ACCOUNT BALANCE SNAPSHOT EXHIBIT"
        ),
        case_caption=(
            "Financial account balances as of July 1, 2026, "
            "with comparative balances at three months, "
            "six months, and one year earlier."
        ),
        folder_id_map=FOLDER_ID_MAP,
    )
)


balance_columns = [
    column
    for column in df_legal_summary.columns
    if column.startswith("Balance as of")
]


display(
    df_legal_summary.style
    .hide(axis="index")
    .format(
        {
            column: "${:,.2f}"
            for column in balance_columns
        },
        na_rep="Not available",
    )
    .set_properties(
        **{
            "text-align": "left",
            "border": "1px solid #9ca3af",
            "padding": "6px",
        }
    )
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("background-color", "#1f2937"),
                    ("color", "white"),
                    ("font-weight", "bold"),
                    ("text-align", "center"),
                    ("border", "1px solid #9ca3af"),
                ],
            }
        ]
    )
)


print(f"\nLegal exhibit saved to:\n{legal_exhibit_path}")

Preparing legal balance exhibit:   0%|          | 0/5 [00:00<?, ?step/s]

Building account exhibits:   0%|          | 0/11 [00:00<?, ?account/s]

Formatting exhibit sheets:   0%|          | 0/11 [00:00<?, ?sheet/s]


LEGAL ACCOUNT BALANCE EXHIBIT CREATED
Accounts included:    11
Summary rows:         11
Balance detail rows:  44
Missing balances:     24
Saved workbook:       /Users/yerik/Documents/___Divorce_2nd_Paper/__inbox/ACCOUNT_BALANCE_EXPORTS/LEGAL_ACCOUNT_BALANCE_EXHIBIT_2026-07-01.xlsx


Registry ID,Financial Institution,Account,Account UID,Account Type,Currency,Balance as of 07/01/2026,Balance as of 04/01/2026,Balance as of 01/01/2026,Balance as of 07/01/2025,Record Coverage,Completeness
A01,OUCU,OUCU Totally Gold Checking (542096-83),ACC_OUCU_CHECKING_83,checking,USD,$892.80,$240.95,"$1,133.67","$1,467.89",01/01/2021 through 07/15/2026,Complete — 4 of 4 balances located
A02,OUCU,OUCU Checking,ACC_OUCU_SHARE_SECURED_70,savings,USD,$500.10,$500.04,$500.03,$500.09,01/31/2021 through 06/30/2026,Complete — 4 of 4 balances located
A03,OUCU,OUCU Checking,ACC_OUCU_GRIZZLY_03,savings,USD,$5.36,$5.36,$608.30,$25.00,12/28/2024 through 03/31/2026,Complete — 4 of 4 balances located
A04,OUCU,OUCU Platinum Visa,ACC_OUCU_VISA_51,credit,USD,Not available,Not available,Not available,Not available,12/06/2020 through 06/26/2026,No reportable balances located
A05,Chase,Chase Total Checking (x4197),ACC_CHASE_CHECKING_4197,checking,USD,"$1,402.90","$2,439.59",Not available,Not available,03/09/2026 through 07/15/2026,Partial — 2 of 4 balances located
A06,Chase,Chase Savings (x8635),ACC_CHASE_SAVINGS_8635,savings,USD,$10.00,$20.00,Not available,Not available,03/16/2026 through 07/14/2026,Partial — 2 of 4 balances located
A07,Chase,Chase Freedom Unlimited Visa (x3262),ACC_CHASE_FREEDOM_3262,credit,USD,Not available,Not available,Not available,Not available,03/21/2026 through 07/15/2026,No reportable balances located
A08,PNC,"PNC Virtual Wallet Spend — JOINT (Soo Jin Kim & Yeriko), x8775",ACC_PNC_JOINT_8775,checking,USD,Not available,Not available,Not available,Not available,10/25/2021 through 07/15/2026,No reportable balances located
A09,PayPal,PayPal — personal,ACC_PAYPAL_PERSONAL,online,USD,$0.00,$0.00,$-3.00,$-970.00,01/06/2021 through 06/28/2026,Complete — 4 of 4 balances located
A10,Robinhood,Robinhood brokerage (705587533),ACC_ROBINHOOD,investment,USD,Not available,Not available,Not available,Not available,04/05/2022 through 07/08/2026,No reportable balances located



Legal exhibit saved to:
/Users/yerik/Documents/___Divorce_2nd_Paper/__inbox/ACCOUNT_BALANCE_EXPORTS/LEGAL_ACCOUNT_BALANCE_EXHIBIT_2026-07-01.xlsx
